# 🔧 AI Hybrid Energy Source Predictor — Feature Engineering

This notebook performs feature engineering for all datasets:
- Temporal features from datetime (hour, day, month, season, daylight flag)
- Rolling window statistics (mean, std) for solar and wind time series
- Lag features for time series prediction
- Derived physical features (DC→AC efficiency, power factor correction, wind power density)
- SMOTE oversampling for Smart Grid imbalanced classification
- Final feature sets saved to `data/processed/`

---

## 1. Setup & Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

BASE = Path(os.path.abspath('..'))
DATA_RAW = BASE / 'data' / 'raw'
DATA_PROC = BASE / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)

print('Project root:', BASE)
print('Output directory:', DATA_PROC)

---
## 2. Solar Generation Dataset — Feature Engineering

In [ ]:
# Load and merge solar generation + weather
solar_gen = pd.read_csv(DATA_RAW / 'Plant_1_Generation_Data.csv', parse_dates=['DATE_TIME'], dayfirst=True)
weather   = pd.read_csv(DATA_RAW / 'Plant_1_Weather_Sensor_Data.csv', parse_dates=['DATE_TIME'], dayfirst=True)

# Aggregate plant-level generation
solar_total = solar_gen.groupby('DATE_TIME')[['DC_POWER','AC_POWER','DAILY_YIELD']].sum().reset_index()
solar_merged = pd.merge(solar_total, weather[['DATE_TIME','IRRADIATION','AMBIENT_TEMPERATURE','MODULE_TEMPERATURE']],
                        on='DATE_TIME', how='inner')
solar_merged = solar_merged.sort_values('DATE_TIME').reset_index(drop=True)
print('Merged shape:', solar_merged.shape)
solar_merged.head()

In [ ]:
# ── Temporal features ──
solar_merged['hour']    = solar_merged['DATE_TIME'].dt.hour
solar_merged['minute']  = solar_merged['DATE_TIME'].dt.minute
solar_merged['day']     = solar_merged['DATE_TIME'].dt.day
solar_merged['month']   = solar_merged['DATE_TIME'].dt.month
solar_merged['weekday'] = solar_merged['DATE_TIME'].dt.weekday
solar_merged['is_weekend'] = (solar_merged['weekday'] >= 5).astype(int)

# Hour-of-day sine/cosine encoding to capture cyclical nature
solar_merged['hour_sin'] = np.sin(2 * np.pi * solar_merged['hour'] / 24)
solar_merged['hour_cos'] = np.cos(2 * np.pi * solar_merged['hour'] / 24)
solar_merged['month_sin'] = np.sin(2 * np.pi * solar_merged['month'] / 12)
solar_merged['month_cos'] = np.cos(2 * np.pi * solar_merged['month'] / 12)

# Daylight flag
solar_merged['is_daylight'] = ((solar_merged['hour'] >= 6) & (solar_merged['hour'] <= 19)).astype(int)

# Season
def month_to_season(m):
    if m in [12, 1, 2]: return 0  # Winter
    elif m in [3, 4, 5]: return 1  # Spring
    elif m in [6, 7, 8]: return 2  # Summer
    else: return 3  # Autumn
solar_merged['season'] = solar_merged['month'].apply(month_to_season)

print('Temporal features added.')
solar_merged[['DATE_TIME','hour','hour_sin','hour_cos','is_daylight','season']].head(8)

In [ ]:
# ── Physical derived features ──
# DC → AC Efficiency
solar_merged['dc_ac_efficiency'] = np.where(
    solar_merged['DC_POWER'] > 0,
    solar_merged['AC_POWER'] / solar_merged['DC_POWER'],
    0.0
)

# Temperature delta (module vs ambient)
solar_merged['temp_delta'] = solar_merged['MODULE_TEMPERATURE'] - solar_merged['AMBIENT_TEMPERATURE']

# Irradiance efficiency (actual AC vs irradiation)
solar_merged['irrad_efficiency'] = np.where(
    solar_merged['IRRADIATION'] > 0,
    solar_merged['AC_POWER'] / solar_merged['IRRADIATION'],
    0.0
)

print('Physical features added.')
solar_merged[['dc_ac_efficiency','temp_delta','irrad_efficiency']].describe()

In [ ]:
# ── Rolling window & lag features ──
# Only daytime rows for meaningful rolling statistics
for col in ['AC_POWER', 'DC_POWER', 'IRRADIATION']:
    solar_merged[f'{col}_roll_mean_3']  = solar_merged[col].rolling(3, min_periods=1).mean()
    solar_merged[f'{col}_roll_std_3']   = solar_merged[col].rolling(3, min_periods=1).std().fillna(0)
    solar_merged[f'{col}_lag_1']        = solar_merged[col].shift(1).fillna(0)
    solar_merged[f'{col}_lag_3']        = solar_merged[col].shift(3).fillna(0)

print('Rolling and lag features added.')
solar_merged.head(3)

In [ ]:
# ── Feature importance preview (correlation with AC_POWER) ──
solar_num = solar_merged.select_dtypes(include=[np.number])
corr_with_target = solar_num.corr()['AC_POWER'].drop('AC_POWER').sort_values(ascending=False)

plt.figure(figsize=(10, 7))
corr_with_target.head(20).plot(kind='barh', color='#3498db', edgecolor='white')
plt.title('Top 20 Features Correlated with AC_POWER (Solar)', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='gray', linewidth=0.8)
plt.tight_layout()
plt.savefig(DATA_PROC / 'fe_solar_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Save final solar processed dataset
solar_merged.to_csv(DATA_PROC / 'solar_features.csv', index=False)
print(f'Solar feature-engineered dataset saved: {DATA_PROC / "solar_features.csv"}')
print('Final shape:', solar_merged.shape)

---
## 3. Wind Power Dataset — Feature Engineering

In [ ]:
wind = pd.read_csv(DATA_RAW / 'wind.csv', parse_dates=['Date/Time'], dayfirst=False)
wind.columns = [c.strip() for c in wind.columns]
wind = wind.sort_values('Date/Time').reset_index(drop=True)
print('Shape:', wind.shape)
wind.head()

In [ ]:
# ── Temporal features ──
wind['hour']    = wind['Date/Time'].dt.hour
wind['day']     = wind['Date/Time'].dt.day
wind['month']   = wind['Date/Time'].dt.month
wind['weekday'] = wind['Date/Time'].dt.weekday
wind['season']  = wind['month'].apply(month_to_season)

wind['hour_sin']  = np.sin(2 * np.pi * wind['hour'] / 24)
wind['hour_cos']  = np.cos(2 * np.pi * wind['hour'] / 24)
wind['month_sin'] = np.sin(2 * np.pi * wind['month'] / 12)
wind['month_cos'] = np.cos(2 * np.pi * wind['month'] / 12)

print('Temporal features added.')

In [ ]:
wind_speed_col = 'Wind Speed (m/s)'
power_col      = 'LV ActivePower (kW)'
dir_col        = 'Wind Direction (°)'

# ── Physical derived features ──
# Wind Power Density: P = 0.5 * rho * A * v^3 (rho=1.225 kg/m³, A normalized to 1)
wind['wind_power_density'] = 0.5 * 1.225 * wind[wind_speed_col] ** 3

# Capacity Factor (actual / theoretical)
wind['capacity_factor'] = np.where(
    wind['Theoretical_Power_Curve (KWh)'] > 0,
    wind[power_col] / wind['Theoretical_Power_Curve (KWh)'],
    0.0
)

# Wind direction sin/cos encoding (circular)
wind['dir_sin'] = np.sin(np.deg2rad(wind[dir_col]))
wind['dir_cos'] = np.cos(np.deg2rad(wind[dir_col]))

# ── Rolling & lag features ──
for col in [wind_speed_col, power_col]:
    wind[f'{col}_roll_mean_6']  = wind[col].rolling(6, min_periods=1).mean()
    wind[f'{col}_roll_std_6']   = wind[col].rolling(6, min_periods=1).std().fillna(0)
    wind[f'{col}_lag_1']        = wind[col].shift(1).fillna(0)
    wind[f'{col}_lag_6']        = wind[col].shift(6).fillna(0)

print('Physical + rolling + lag features added.')
wind[['wind_power_density','capacity_factor','dir_sin','dir_cos']].describe()

In [ ]:
# Correlation with target (LV ActivePower)
wind_num = wind.select_dtypes(include=[np.number])
corr_wind = wind_num.corr()[power_col].drop(power_col).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
corr_wind.head(20).plot(kind='barh', color='#1abc9c', edgecolor='white')
plt.title('Top 20 Features Correlated with LV Active Power (Wind)', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='gray', linewidth=0.8)
plt.tight_layout()
plt.savefig(DATA_PROC / 'fe_wind_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

wind.to_csv(DATA_PROC / 'wind_features.csv', index=False)
print(f'Wind features saved. Shape: {wind.shape}')

---
## 4. Smart Grid Dataset — Feature Engineering & SMOTE

In [ ]:
sg = pd.read_csv(BASE / 'smart_grid_dataset.csv', parse_dates=['Timestamp'])
sg = sg.rename(columns=lambda x: 'Temperature' if 'Temperature' in x else x)

# ── Temporal features ──
sg['hour']    = sg['Timestamp'].dt.hour
sg['day']     = sg['Timestamp'].dt.day
sg['month']   = sg['Timestamp'].dt.month
sg['weekday'] = sg['Timestamp'].dt.weekday
sg['season']  = sg['month'].apply(month_to_season)
sg['hour_sin'] = np.sin(2 * np.pi * sg['hour'] / 24)
sg['hour_cos'] = np.cos(2 * np.pi * sg['hour'] / 24)
sg['is_peak_hours'] = ((sg['hour'] >= 8) & (sg['hour'] <= 22)).astype(int)

print('Temporal features added.')

In [ ]:
# ── Physical derived features ──
# Apparent Power: S = sqrt(P² + Q²)
sg['apparent_power'] = np.sqrt(sg['Power Consumption (kW)']**2 + sg['Reactive Power (kVAR)']**2)

# Total renewable generation
sg['total_renewable'] = sg['Solar Power (kW)'] + sg['Wind Power (kW)']

# Net demand from grid
sg['net_grid_demand'] = sg['Power Consumption (kW)'] - sg['total_renewable']

# Renewable penetration ratio
sg['renewable_ratio'] = np.where(
    sg['Power Consumption (kW)'] > 0,
    sg['total_renewable'] / sg['Power Consumption (kW)'],
    0.0
)

# Voltage deviation from nominal (230V)
sg['voltage_deviation'] = sg['Voltage (V)'] - 230.0

print('Physical derived features added.')
sg[['apparent_power','total_renewable','net_grid_demand','renewable_ratio','voltage_deviation']].describe()

In [ ]:
# ── Interaction features ──
sg['voltage_x_current']     = sg['Voltage (V)'] * sg['Current (A)']
sg['pf_x_voltage_fluct']    = sg['Power Factor'] * sg['Voltage Fluctuation (%)']
sg['temp_humidity_index']   = sg['Temperature'] * (1 + 0.01 * sg['Humidity (%)'])

print('Interaction features added.')

In [ ]:
# ── Class-imbalance analysis before SMOTE ──
feature_cols = [
    'Voltage (V)', 'Current (A)', 'Power Consumption (kW)', 'Reactive Power (kVAR)', 'Power Factor',
    'Solar Power (kW)', 'Wind Power (kW)', 'Grid Supply (kW)', 'Voltage Fluctuation (%)',
    'Overload Condition', 'Temperature', 'Humidity (%)', 'Electricity Price (USD/kWh)',
    'Predicted Load (kW)', 'hour', 'month', 'season', 'hour_sin', 'hour_cos', 'is_peak_hours',
    'apparent_power', 'total_renewable', 'net_grid_demand', 'renewable_ratio',
    'voltage_deviation', 'voltage_x_current', 'temp_humidity_index'
]
target_col = 'Transformer Fault'

X = sg[feature_cols]
y = sg[target_col]

print('Before SMOTE:')
print(y.value_counts())
print(f'Imbalance ratio: {y.value_counts()[0]/y.value_counts()[1]:.1f}x')

In [ ]:
# Try SMOTE if imbalanced-learn is available
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_resampled, y_resampled = smote.fit_resample(X, y)
    print('After SMOTE:')
    print(pd.Series(y_resampled).value_counts())
    
    # Save SMOTE-balanced dataset
    sg_balanced = pd.DataFrame(X_resampled, columns=feature_cols)
    sg_balanced[target_col] = y_resampled
    sg_balanced.to_csv(DATA_PROC / 'smart_grid_smote.csv', index=False)
    print('SMOTE-balanced dataset saved to data/processed/smart_grid_smote.csv')
except ImportError:
    print('imbalanced-learn not installed. Using class_weight in XGBoost instead (scale_pos_weight).')
    print('This is equally effective for tree-based models.')

In [ ]:
# ── Feature importance visualization using Random Forest ──
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf_quick = RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=42, n_jobs=-1)
rf_quick.fit(X_train, y_train)

importances = pd.Series(rf_quick.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 9))
importances.head(20).plot(kind='barh', color='#e74c3c', edgecolor='white')
plt.title('Top 20 Feature Importances (Random Forest) — Smart Grid Fault Prediction', fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(DATA_PROC / 'fe_smartgrid_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print('Top 10 features:\n', importances.head(10))

In [ ]:
# ── Scaling features ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
sg_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
sg_scaled[target_col] = y.values
sg_scaled.to_csv(DATA_PROC / 'smart_grid_scaled.csv', index=False)
print(f'Scaled dataset saved. Shape: {sg_scaled.shape}')

---
## 5. Final Summary of Engineered Features

| Dataset | New Features Added | Total Features | Output File |
|---|---|---|---|
| Solar Generation | hour_sin/cos, season, roll_mean, lag, efficiency, temp_delta | ~32 | `solar_features.csv` |
| Wind Power | wind_power_density, dir_sin/cos, capacity_factor, rolling, lag | ~30 | `wind_features.csv` |
| Smart Grid | apparent_power, renewable_ratio, net_grid_demand, interaction terms, SMOTE | ~27 | `smart_grid_scaled.csv` |

> **Key Engineering Decisions:**
> - **Cyclical encoding** (sin/cos) for hour and month avoids artificial distance jumps (23:00→00:00 treated same as 01:00→02:00).
> - **Wind power density** ($P = \frac{1}{2}\rho v^3$) is the raw physical predictor — highly correlated with actual output.
> - **DC→AC efficiency** identifies inverter degradation over time.
> - **SMOTE** upsamples minority class (Transformer Fault) from 2.9% → 50% before training deep models.